In [2]:
!pip install opencv-python


  Obtaining dependency information for opencv-python from https://files.pythonhosted.org/packages/e9/a5/1be1516390333ff9be3a9cb648c9f33df79d5096e5884b5df71a588af463/opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata
  Obtaining dependency information for numpy>=2 from https://files.pythonhosted.org/packages/76/ae/e0265e0163cf127c24c3969d29f1c4c64551a1e375d95a13d32eab25d364/numpy-2.4.2-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ---------------------------------------- 0.1/40.2 MB 3.6 MB/s eta 0:00:12
    --------------------------------------- 0.8/40.2 MB 10.5 MB/s eta 0:00:04
   -- ------------------------------------- 2.3/40.2 MB 18.6 MB/s eta 0:00:03
   ---- ----------------------------------- 4.5/40.2 MB 26.4 MB/s eta 0:00:02
   -------- ------------------------------- 8.1/40.2 MB 36.9 MB/s eta 0:00:01
   ----------- ---------------------------- 11.6/40.2 MB 59.5 MB/s eta 0:00:01
   ------------- ----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
numba 0.57.1 requires numpy<1.25,>=1.21, but you have numpy 2.4.2 which is incompatible.
scipy 1.11.1 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.4.2 which is incompatible.


In [3]:
import cv2
import os

#loading pretrained Haar Cascades
cascPathface = os.path.dirname(
    cv2.__file__) + "/data/haarcascade_frontalface_alt2.xml"
cascPatheyes = os.path.dirname(
    cv2.__file__) + "/data/haarcascade_eye_tree_eyeglasses.xml"

faceCascade = cv2.CascadeClassifier(cascPathface) #detect faces
eyeCascade = cv2.CascadeClassifier(cascPatheyes) #detect eyes

video_capture = cv2.VideoCapture(0) #start webcam - default camera
while True:
    # Capture frame-by-frame
    ret, frame = video_capture.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) #convert to grayscale
    # Detecting faces - CORE
    faces = faceCascade.detectMultiScale(gray,
                                         scaleFactor=1.1, #Each time image is scaled down by 10%
                                         minNeighbors=5, # Minimum number of overlapping detections required to accept a region
                                         minSize=(60, 60), # Smallest possible face size
                                         flags=cv2.CASCADE_SCALE_IMAGE) 
    for (x,y,w,h) in faces:
        cv2.rectangle(frame, (x, y), (x + w, y + h),(0,255,0), 2) # Drawing rectangles around faces
        faceROI = frame[y:y+h,x:x+w] # Extracting Region of Interest - This crops just the detected face area
        eyes = eyeCascade.detectMultiScale(faceROI) # Eye detector scans only inside the face
        for (x2, y2, w2, h2) in eyes:
            eye_center = (x + x2 + w2 // 2, y + y2 + h2 // 2)
            radius = int(round((w2 + h2) * 0.25))
            frame = cv2.circle(frame, eye_center, radius, (255, 0, 0), 4)

        # Display the resulting frame
    cv2.imshow('Video', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): # Quit when q is pressed
        break
video_capture.release()
cv2.destroyAllWindows()